In [44]:
import requests
from pydantic import BaseModel

In [45]:
class SearchResult(BaseModel):
    page_id: int
    title: str
    snippet: str


class WikipediaPage(BaseModel):
    page_id: int
    title: str
    url: str
    content: str


In [58]:
BASE_URL = "https://en.wikipedia.org/w/api.php"
headers = {"User-Agent": "QuizApp/0.1 (abc@example.com)"}


def search(query: str, limit: int = 5) -> list[SearchResult]:
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": limit,
    }

    response = requests.get(BASE_URL, headers=headers, params=params)
    print(response)
    response.raise_for_status()

    data = response.json()["query"]["search"]

    return [
        SearchResult(
            page_id=result["pageid"],
            title=result["title"],
            snippet=result["snippet"],
        )
        for result in data
    ]

def get_page(title: str) -> WikipediaPage:
    params = {
        "action": "query",
        "prop": "extracts",
        "titles": title,
        "explaintext": True,
        "format": "json",
        "exsectionformat": "wiki",
        "redirects": 1
    }

    response = requests.get(BASE_URL, headers=headers, params=params)

    pages = response.json()["query"]["pages"]

    page = next(iter(pages.values()))

    return WikipediaPage(
        page_id=page["pageid"],
        title=page["title"],
        url=f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}",
        content=page.get("extract", "")
    )

def resolve_topic_to_article(topic):
    results = search(topic, limit=3)
    if not results:
        return None
    top = results[0]
    content = get_page(top.title)
    return {"title": top.title, "content": content.content}

In [59]:
topic = "Linux"
result = resolve_topic_to_article(topic=topic)

<Response [200]>


In [60]:
import re

BOILERPLATE_SECTIONS = {
    "see also",
    "references",
    "external links",
    "further reading",
    "notes",
    "citations",
    "bibliography",
    "sources",
}


def parse_sections(full_text):
    heading_pattern = re.compile(r"^(=+)\s*(.*?)\s*=+$")
    lines = full_text.split("\n")

    raw_sections = []
    current = {"title": None, "level": 2, "text_lines": []}

    for line in lines:
        match = heading_pattern.match(line.strip())
        if match:
            raw_sections.append(current)
            level = len(match.group(1))
            current = {
                "title": match.group(2).strip(),
                "level": level,
                "text_lines": [],
            }
        else:
            current["text_lines"].append(line)
    raw_sections.append(current)

    stack = []
    sections = []
    for s in raw_sections:
        title = s["title"] or "Introduction"
        if title.lower() in BOILERPLATE_SECTIONS:
            continue

        while stack and stack[-1][0] >= s["level"]:
            stack.pop()
        stack.append((s["level"], title))

        text = "\n".join(l for l in s["text_lines"] if l.strip())
        if not text:
            continue

        sections.append(
            {
                "title": title,  # leaf title — new
                "breadcrumb": " > ".join(t for _, t in stack),
                "text": text,
            }
        )

    return sections


In [61]:
from urllib.parse import quote


def wiki_section_url(article_title, section_title):
    base = f"https://en.wikipedia.org/wiki/{quote(article_title.replace(' ', '_'))}"
    if section_title == "Introduction":
        return base  # intro has no heading, so no anchor to link to
    anchor = quote(section_title.replace(" ", "_"))
    return f"{base}#{anchor}"


In [62]:
sections = parse_sections(result.get("content", ""))

In [63]:
import tiktoken

encoder = tiktoken.get_encoding("cl100k_base")


def count_tokens(text):
    return len(encoder.encode(text))


MIN_TOKENS = 50
MAX_TOKENS = 400


def build_chunks(sections, article_title):
    chunks = []
    buffer = None

    for section in sections:
        tokens = count_tokens(section["text"])

        if tokens < MIN_TOKENS:
            if buffer is None:
                buffer = section
            else:
                buffer["text"] += "\n\n" + section["text"]
            continue

        if buffer is not None:
            chunks.extend(split_if_needed(buffer, article_title))
            buffer = None
        chunks.extend(split_if_needed(section, article_title))

    if buffer is not None:
        chunks.extend(split_if_needed(buffer, article_title))

    return chunks


def split_if_needed(section, article_title):
    text = section["text"]
    if count_tokens(text) <= MAX_TOKENS:
        return [format_chunk(article_title,section["title"], section["breadcrumb"], text)]

    paragraphs = [p for p in text.split("\n\n") if p.strip()]
    sub_chunks, current, current_tokens = [], [], 0

    for para in paragraphs:
        para_tokens = count_tokens(para)
        if current and current_tokens + para_tokens > MAX_TOKENS:
            sub_chunks.append("\n\n".join(current))
            current = [current[-1], para]  # 1-paragraph overlap
            current_tokens = count_tokens(current[0]) + para_tokens
        else:
            current.append(para)
            current_tokens += para_tokens

    if current:
        sub_chunks.append("\n\n".join(current))

    return [format_chunk(article_title, section["title"], section["breadcrumb"], sc) for sc in sub_chunks]


def format_chunk(article_title, section_title, breadcrumb, text):
    return {
        "text": f"Article: {article_title}\nSection: {breadcrumb}\n\n{text}",
        "article_title": article_title,
        "section_title": section_title,
        "section_breadcrumb": breadcrumb,
        "source_url": wiki_section_url(article_title, section_title),
        "raw_text": text,
    }


In [64]:
chunks = build_chunks(sections=sections, article_title=result.get("title", ""))

In [65]:
chunks[2]

{'text': 'Article: Linux\nSection: History > Precursors\n\nThe Unix operating system was conceived of and implemented in 1969, at AT&T\'s Bell Labs in the United States, by Ken Thompson, Dennis Ritchie, Douglas McIlroy, and Joe Ossanna. First released in 1971, Unix was written entirely in assembly language, as was common practice at the time. In 1973, in a key pioneering approach, it was rewritten in the C programming language by Dennis Ritchie (except for some hardware and I/O routines). The availability of a high-level language implementation of Unix made its porting to different computer platforms easier.\nAs a 1956 antitrust case forbade AT&T from entering the computer business, AT&T provided the operating system\'s source code to anyone who asked. As a result, Unix use grew quickly and it became widely adopted by academic institutions and businesses. In 1984, AT&T divested itself of its regional operating companies, and was released from its obligation not to enter the computer bu

## Embeddings

In [1]:
from google import genai
from google.genai import types
import os

In [2]:
gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [83]:
def get_embedding(text, emb_model="gemini-embedding-2"):
    result = gemini_client.models.embed_content(
        model=emb_model,
        contents=text,
        config=types.EmbedContentConfig(output_dimensionality=1536),
    )


    return result.embeddings[0].values

In [86]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [87]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [88]:
qdrant_client.create_collection(
    collection_name="Quiz-App-Dev-Collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)


True

In [90]:
data_to_embed = [chunks[0]]

In [91]:
pointstructs = []
for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data["text"])
    pointstructs.append(PointStruct(id=i, vector=embedding, payload={
        "article_title": data["article_title"],
        "section_title": data["section_title"],
        "source_url": data["source_url"],
        "raw_text": data["raw_text"]
    }))


In [92]:
qdrant_client.upsert(
    collection_name="Quiz-App-Dev-Collection", wait=True, points=pointstructs
)


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)